# Sarcheshmeh – end-to-end alteration-mapping study (no login, free data)
One notebook = data → methods → validation → exploration targets → paper tables/figures → industrial report.
Run all cells (Runtime → Run all). Everything is saved in `results/` and zipped at the end.
**Optional but strongly recommended for publication:** upload a polygon file of a *reference* (e.g. mapped alteration zones / mineralised zone digitised from the geology map) and set `CFG["ref_path"]`.

In [ ]:
!pip -q install pystac-client odc-stac rasterio geopandas shapely scikit-image scipy seaborn requests
import os
os.environ["AWS_NO_SIGN_REQUEST"] = "YES"; os.environ["AWS_DEFAULT_REGION"] = "eu-central-1"
import json, time, warnings, shutil, base64, platform
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")

CFG = dict(
    bbox=(55.75, 29.85, 56.00, 30.05),      # lon_min, lat_min, lon_max, lat_max  -> check on a map, enlarge if needed
    epsg=32640, res=20, min_obs=3,
    period_A=("2022-02-01", "2024-10-31"),  # training period
    period_B=("2024-11-01", "2026-10-31"),  # independent period for reproducibility tests
    cloud_lt=15, max_scenes=30,
    k=8, seeds=[0, 1, 2, 3, 4], latent=8, epochs=15, n_train=300000,
    primary="SSL_AE",
    ref_path=None,         # e.g. "/content/ref_zones.geojson" (polygons = known altered/mineralised zones)
    mine_mask_path=None,   # optional polygons of pit/tailings/dumps; otherwise OpenStreetMap is tried
    block_km=2.0,          # spatial-CV block size
    min_target_ha=5, top_quantile=0.90, conf_min=0.8,
    out="results")
OUT = CFG["out"]
for d in ["figures", "tables", "rasters", "vectors"]:
    os.makedirs(f"{OUT}/{d}", exist_ok=True)
LOG = []
def log(*a):
    msg = " ".join(str(x) for x in a); print(msg); LOG.append(f"{time.strftime('%H:%M:%S')} {msg}")
BANDS = ["blue", "green", "red", "nir08", "swir16", "swir22"]
FEATS = BANDS + ["clay_B11_B12", "ferric_R_B", "ferrous_B11_B8A", "slope"]
sns.set_theme(style="whitegrid")

## 1. Data (Sentinel-2 L2A + Copernicus DEM, public STAC, two independent time windows)

In [ ]:
def load_all(CFG):
    import xarray as xr
    from pystac_client import Client
    from odc.stac import load
    cat = Client.open("https://earth-search.aws.element84.com/v1")
    bbox = CFG["bbox"]; out = {}
    for tag in ["A", "B"]:
        p0, p1 = CFG["period_" + tag]
        items = list(cat.search(collections=["sentinel-2-l2a"], bbox=bbox, datetime=f"{p0}/{p1}",
                                query={"eo:cloud_cover": {"lt": CFG["cloud_lt"]}}).items())
        log(f"period {tag}: {len(items)} scenes found")
        if not items: raise RuntimeError("no scenes: widen dates / bbox / cloud limit")
        items = items[::max(1, len(items)//CFG["max_scenes"])]
        ds = load(items, bands=BANDS+["scl"], bbox=bbox, resolution=CFG["res"], crs=f"EPSG:{CFG['epsg']}",
                  groupby="solar_day", chunks={"x": 1024, "y": 1024})
        ok = ds["scl"].isin([4, 5])        # vegetation / bare soil only
        comp = xr.Dataset({b: (ds[b].where(ok & (ds[b] > 0))*1e-4 - 0.1).median("time", skipna=True)
                           for b in BANDS}).compute()
        out["R"+tag] = {b: comp[b].values.astype("float32") for b in BANDS}
        out["nobs"+tag] = ok.sum("time").compute().values
        out["n_scenes"+tag] = len(items)
        if tag == "A": out["geobox"] = comp.odc.geobox; ref = comp
        log(f"period {tag}: composite from {len(items)} dates")
    dem_items = list(cat.search(collections=["cop-dem-glo-30"], bbox=bbox).items())
    out["dem"] = load(dem_items, bands=["data"], like=ref, chunks={}).squeeze()["data"].compute().values.astype("float32")
    return out

def osm_mine_mask(CFG, shape, geobox):
    """Pits / quarries / industrial-mine polygons from OpenStreetMap (best effort)."""
    try:
        import requests, shapely.geometry as sg, rasterio.features
        from pyproj import Transformer
        w, s, e, n = CFG["bbox"]
        tags = ['["landuse"="quarry"]', '["industrial"="mine"]', '["man_made"="tailings_pond"]', '["landuse"="industrial"]']
        q = "[out:json][timeout:60];(" + "".join(f"way{t}({s},{w},{n},{e});" for t in tags) + ");out geom;"
        r = requests.post("https://overpass-api.de/api/interpreter", data={"data": q}, timeout=90)
        tr = Transformer.from_crs(4326, CFG["epsg"], always_xy=True); geoms = []
        for el in r.json().get("elements", []):
            g = el.get("geometry", [])
            if len(g) >= 4:
                xy = [tr.transform(p["lon"], p["lat"]) for p in g]
                geoms.append(sg.Polygon(xy).buffer(100))
        if not geoms: return np.zeros(shape, bool), 0
        m = rasterio.features.rasterize([(g, 1) for g in geoms], out_shape=shape, transform=geobox.transform, fill=0, dtype="uint8")
        return m.astype(bool), len(geoms)
    except Exception as ex:
        log("OSM mine mask failed:", repr(ex)[:120]); return np.zeros(shape, bool), 0

def file_mine_mask(path, shape, geobox, epsg):
    import geopandas as gpd, rasterio.features
    g = gpd.read_file(path).to_crs(epsg)
    return rasterio.features.rasterize([(x, 1) for x in g.geometry], out_shape=shape, transform=geobox.transform, fill=0, dtype="uint8").astype(bool)

data = load_all(CFG)
log("grid:", data["RA"]["blue"].shape)

## 2. Features, validity masks and disturbance mask

In [ ]:
def build_X(R, slope):
    b = {k: np.clip(v, 1e-3, 1.5) for k, v in R.items()}
    clay = b["swir16"]/b["swir22"]; ferric = b["red"]/b["blue"]; ferrous = b["swir16"]/b["nir08"]
    X = np.stack([b[k] for k in BANDS] + [clay, ferric, ferrous, slope], -1).astype("float32")
    return X

dem = data["dem"]; gy, gx = np.gradient(dem, CFG["res"]); slope = np.degrees(np.arctan(np.hypot(gx, gy))).astype("float32")
XA = build_X(data["RA"], slope); XB = build_X(data["RB"], slope)
H, W, NF = XA.shape
if CFG["mine_mask_path"]:
    mine = file_mine_mask(CFG["mine_mask_path"], (H, W), data["geobox"], CFG["epsg"]); nmine = -1
else:
    mine, nmine = osm_mine_mask(CFG, (H, W), data["geobox"]) if data.get("geobox") is not None else (np.zeros((H, W), bool), 0)
log("mining-disturbance mask: polygons =", nmine, "| masked fraction =", round(mine.mean(), 4))
validA = np.isfinite(XA).all(-1) & (data["nobsA"] >= CFG["min_obs"]) & ~mine
validB = np.isfinite(XB).all(-1) & (data["nobsB"] >= CFG["min_obs"]) & ~mine
common = validA & validB
log(f"valid px  A={validA.mean():.3f}  B={validB.mean():.3f}  A&B={common.mean():.3f}")
XAf, XBf = XA.reshape(-1, NF), XB.reshape(-1, NF)
idxA, idxB = np.flatnonzero(validA.ravel()), np.flatnonzero(validB.ravel())

## 3. Methods: baselines vs self-supervised masked autoencoder (SSL)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans

class AEWrap:
    def __init__(s, net, dev): s.net, s.dev = net, dev
    def embed(s, Z):
        import torch
        out = []
        with torch.no_grad():
            for i in range(0, len(Z), 16384):
                out.append(s.net.enc(torch.tensor(Z[i:i+16384], dtype=torch.float32).to(s.dev)).cpu().numpy())
        return np.concatenate(out)

def train_ae(Z, seed, latent, epochs, mask_ratio=0.3):
    try:
        import torch, torch.nn as nn
    except ImportError:
        return None
    torch.manual_seed(seed); np.random.seed(seed)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    d = Z.shape[1]
    class Net(nn.Module):
        def __init__(s):
            super().__init__()
            s.enc = nn.Sequential(nn.Linear(d, 128), nn.GELU(), nn.Linear(128, 32), nn.GELU(), nn.Linear(32, latent))
            s.dec = nn.Sequential(nn.Linear(latent, 32), nn.GELU(), nn.Linear(32, 128), nn.GELU(), nn.Linear(128, d))
        def forward(s, x): return s.dec(s.enc(x))
    net = Net().to(dev); opt = torch.optim.AdamW(net.parameters(), 2e-3, weight_decay=1e-4)
    T = torch.tensor(Z, dtype=torch.float32)
    for ep in range(epochs):
        perm = torch.randperm(len(T))
        for i in range(0, len(T), 2048):
            x = T[perm[i:i+2048]].to(dev)
            keep = (torch.rand_like(x) > mask_ratio).float()
            loss = (((net(x*keep)-x)**2)*(1-keep)).sum()/(1-keep).sum().clamp(min=1)
            opt.zero_grad(); loss.backward(); opt.step()
    net.eval(); return AEWrap(net, dev)

METHODS = {
    "KMeans_bands":  dict(cols=list(range(6)), kind="raw"),
    "KMeans_ratios": dict(cols=[6, 7, 8],      kind="raw"),
    "PCA3":          dict(cols=list(range(9)), kind="pca"),
    "SSL_AE":        dict(cols=list(range(9)), kind="ssl"),
    "SSL_AE_DEM":    dict(cols=list(range(10)), kind="ssl"),
}
AE_OK = True

class Method:
    def __init__(s, name, seed):
        s.name, s.seed = name, seed; s.cfg = METHODS[name]
    def fit(s, X):
        global AE_OK
        s.sc = StandardScaler().fit(X[:, s.cfg["cols"]]); Z = s.sc.transform(X[:, s.cfg["cols"]]).astype("float32")
        k = s.cfg["kind"]
        if k == "pca": s.pca = PCA(3, random_state=s.seed).fit(Z)
        if k == "ssl":
            s.ae = train_ae(Z, s.seed, CFG["latent"], CFG["epochs"])
            if s.ae is None:
                AE_OK = False; s.pca = PCA(CFG["latent"], random_state=s.seed).fit(Z)
        F = s.transform_Z(Z)
        s.km = MiniBatchKMeans(CFG["k"], random_state=s.seed, n_init=5, batch_size=8192).fit(F)
        return s
    def transform_Z(s, Z):
        k = s.cfg["kind"]
        if k == "raw": return Z
        if k == "pca": return s.pca.transform(Z)
        return s.ae.embed(Z) if s.ae is not None else s.pca.transform(Z)
    def features(s, X):
        return s.transform_Z(s.sc.transform(X[:, s.cfg["cols"]]).astype("float32"))
    def predict_grid(s, Xf, idx, shape):
        g = np.full(shape[0]*shape[1], -1, np.int8)
        for i in range(0, len(idx), 300000):
            j = idx[i:i+300000]; g[j] = s.km.predict(s.features(Xf[j]))
        return g.reshape(shape)

## 4. Experiments (every method × several seeds, fitted on period A, applied to independent period B)

In [ ]:
rng = np.random.default_rng(0)
G, MODELS = {}, {}
t0 = time.time()
for m in METHODS:
    for sd in CFG["seeds"]:
        r = np.random.default_rng(sd)
        sel = r.choice(idxA, min(CFG["n_train"], len(idxA)), replace=False)
        mod = Method(m, sd).fit(XAf[sel])
        G[(m, sd, "A")] = mod.predict_grid(XAf, idxA, (H, W)); G[(m, sd, "B")] = mod.predict_grid(XBf, idxB, (H, W))
        if sd == CFG["seeds"][0]: MODELS[m] = mod
    log(f"{m} done  ({time.time()-t0:.0f}s)")
if not AE_OK: log("WARNING: torch not available -> SSL methods fell back to PCA embeddings")

## 5. Label-free evaluation: reproducibility, stability, coherence, separability

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score

def coherence(g):
    v1 = (g[:, :-1] >= 0) & (g[:, 1:] >= 0); v2 = (g[:-1] >= 0) & (g[1:] >= 0)
    return float(((g[:, :-1] == g[:, 1:]) & v1).sum() + ((g[:-1] == g[1:]) & v2).sum()) / float(v1.sum() + v2.sum())

scA = StandardScaler().fit(XAf[idxA][:, :9])
sub = np.random.default_rng(1).choice(idxA, min(15000, len(idxA)), replace=False)
ZC = scA.transform(XAf[sub][:, :9])
rows = []
for m in METHODS:
    tc, co, si, db = [], [], [], []
    for sd in CFG["seeds"]:
        gA, gB = G[(m, sd, "A")], G[(m, sd, "B")]; cm = (gA >= 0) & (gB >= 0)
        tc.append(float((gA[cm] == gB[cm]).mean())); co.append(coherence(gA))
        lab = gA.ravel()[sub]; si.append(float(silhouette_score(ZC, lab))); db.append(float(davies_bouldin_score(ZC, lab)))
    aris = [adjusted_rand_score(G[(m, a, "A")].ravel()[sub], G[(m, b, "A")].ravel()[sub])
            for i, a in enumerate(CFG["seeds"]) for b in CFG["seeds"][i+1:]]
    rows.append(dict(method=m, temporal_consistency=np.mean(tc), temporal_sd=np.std(tc), seed_ARI=np.mean(aris) if aris else np.nan,
                     spatial_coherence=np.mean(co), silhouette=np.mean(si), davies_bouldin=np.mean(db)))
T1 = pd.DataFrame(rows).set_index("method").round(4)
T1.to_csv(f"{OUT}/tables/T1_method_comparison.csv"); T1

## 6. Geological interpretation of clusters, confidence map and exploration targets

In [ ]:
from scipy import ndimage
P = CFG["primary"]; s0 = CFG["seeds"][0]
gP = G[(P, s0, "A")]; flat = gP.ravel()
pix = XAf[idxA]; mu_ = pix.mean(0); sd_ = pix.std(0) + 1e-9
sig = []
for c in range(CFG["k"]):
    mk = flat == c
    if mk.sum() == 0: continue
    v = XAf[mk].mean(0); z = (v - mu_)/sd_
    zc, zf = z[6], z[7]
    role = ("clay/Al-OH + iron-oxide" if zc > .5 and zf > .5 else "clay/Al-OH-rich (sericite/argillic proxy)" if zc > .5 else
            "iron-oxide-rich (gossan/oxidation proxy)" if zf > .5 else "background / low-alteration" if zc < -.2 else "intermediate")
    sig.append(dict(cluster=c, area_ha=mk.sum()*CFG["res"]**2/1e4, role=role, **{f: v[i] for i, f in enumerate(FEATS)},
                    **{"z_"+f: z[i] for i, f in enumerate(FEATS)}))
T2 = pd.DataFrame(sig).set_index("cluster").round(3); T2.to_csv(f"{OUT}/tables/T2_cluster_signatures.csv")

def clay_clusters(g):
    f = g.ravel(); zs = {}
    for c in range(CFG["k"]):
        mk = f == c
        if mk.sum(): zs[c] = (XAf[mk][:, 6].mean() - mu_[6])/sd_[6]
    cs = [c for c, z in zs.items() if z > .5]
    return cs if cs else [max(zs, key=zs.get)]

conf_sum = np.zeros((H, W), np.float32); conf_n = 0
for sd in CFG["seeds"]:
    cs = clay_clusters(G[(P, sd, "A")])      # cluster identity decided on period A, same model applied to B
    for per in ["A", "B"]:
        g = G[(P, sd, per)]; conf_sum += np.isin(g, cs).astype(np.float32); conf_n += 1
conf = np.where(common, conf_sum/conf_n, np.nan).astype("float32")

def rz(x):
    m = np.nanmedian(x); s = 1.4826*np.nanmedian(np.abs(x - m)) + 1e-9; return (x - m)/s
clayM = np.nanmean([XA[..., 6], XB[..., 6]], 0); ferM = np.nanmean([XA[..., 7], XB[..., 7]], 0)
AI = (rz(np.where(common, clayM, np.nan)) + 0.5*rz(np.where(common, ferM, np.nan))).astype("float32")
thr = np.nanquantile(AI[common], CFG["top_quantile"])
cand = (conf >= CFG["conf_min"]) & (AI >= thr) & common
cand = ndimage.binary_opening(cand, structure=np.ones((2, 2)))
lab_, nlab = ndimage.label(cand, structure=np.ones((3, 3)))
minpx = int(CFG["min_target_ha"]*1e4/CFG["res"]**2)
cnt = np.bincount(lab_.ravel(), minlength=nlab+1)
keep = [i for i in range(1, nlab+1) if cnt[i] >= minpx]
tr_ = None
try:
    from pyproj import Transformer; tr_ = Transformer.from_crs(CFG["epsg"], 4326, always_xy=True)
except Exception: pass
gb = data.get("geobox"); trf = gb.transform if gb is not None else None
trows = []
for i in keep:
    cy, cx = ndimage.center_of_mass(lab_ == i)
    X0 = (trf.c + (cx+.5)*trf.a, trf.f + (cy+.5)*trf.e) if trf is not None else (np.nan, np.nan)
    ll = tr_.transform(*X0) if (tr_ is not None and trf is not None) else (np.nan, np.nan)
    trows.append(dict(id=i, area_ha=cnt[i]*CFG["res"]**2/1e4, mean_AI=float(np.nanmean(AI[lab_ == i])),
                      mean_conf=float(np.nanmean(conf[lab_ == i])), mean_clay=float(np.nanmean(clayM[lab_ == i])),
                      mean_ferric=float(np.nanmean(ferM[lab_ == i])), mean_slope=float(np.nanmean(slope[lab_ == i])),
                      lon=ll[0], lat=ll[1]))
T4 = pd.DataFrame(trows)
if len(T4):
    T4["score"] = T4.mean_AI*T4.mean_conf; T4 = T4.sort_values("score", ascending=False).reset_index(drop=True); T4["rank"] = T4.index+1
T4.round(4).to_csv(f"{OUT}/tables/T4_exploration_targets.csv", index=False)
log(f"targets: {len(T4)} (>= {CFG['min_target_ha']} ha, conf >= {CFG['conf_min']}, top {int((1-CFG['top_quantile'])*100)}% AI)")

def write_raster(name, arr, nodata):
    try:
        import rasterio
        with rasterio.open(f"{OUT}/rasters/{name}.tif", "w", driver="GTiff", height=H, width=W, count=1, dtype=arr.dtype,
                           crs=str(gb.crs), transform=gb.transform, nodata=nodata) as d: d.write(arr, 1)
    except Exception:
        np.save(f"{OUT}/rasters/{name}.npy", arr)
write_raster("cluster_map_A_primary", gP.astype("int16"), -1); write_raster("confidence", conf, np.nan)
write_raster("alteration_index", AI, np.nan); write_raster("clay_ratio_mean", clayM.astype("float32"), np.nan)
write_raster("ferric_ratio_mean", ferM.astype("float32"), np.nan); write_raster("targets_mask", lab_.astype("int32"), 0)
try:
    import rasterio.features, geopandas as gpd, shapely.geometry as sg
    polys = [(sg.shape(g), int(v)) for g, v in rasterio.features.shapes(np.where(np.isin(lab_, keep), lab_, 0).astype("int32"),
             mask=np.isin(lab_, keep), transform=gb.transform)]
    gdf = gpd.GeoDataFrame({"id": [p[1] for p in polys]}, geometry=[p[0] for p in polys], crs=CFG["epsg"]).merge(T4, on="id")
    gdf.to_crs(4326).to_file(f"{OUT}/vectors/exploration_targets.geojson", driver="GeoJSON")
except Exception as ex:
    log("vector export skipped:", repr(ex)[:100])

## 7. Supervised validation with spatial block CV (needs a reference polygon file)

In [ ]:
T3 = None; IMP = None
ref_ok = False
if CFG["ref_path"]:
    try:
        import geopandas as gpd, rasterio.features
        g = gpd.read_file(CFG["ref_path"]).to_crs(CFG["epsg"])
        yref = rasterio.features.rasterize([(x, 1) for x in g.geometry], out_shape=(H, W), transform=gb.transform, fill=0, dtype="uint8")
        ref_ok = True
    except Exception as ex:
        log("reference load failed:", repr(ex)[:120])
if ref_ok:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import StratifiedKFold, GroupKFold
    from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
    from sklearn.inspection import permutation_importance
    rr = np.random.default_rng(0); y_all = yref.ravel()[idxA]
    pos = np.flatnonzero(y_all == 1); neg = np.flatnonzero(y_all == 0)
    log(f"reference pixels: positives={len(pos)} ({100*len(pos)/len(idxA):.2f}% of valid)")
    sel = np.concatenate([rr.choice(pos, min(len(pos), 15000), replace=False), rr.choice(neg, min(len(neg), 30000), replace=False)])
    pidx = idxA[sel]; y = y_all[sel]
    rows_, cols_ = pidx//W, pidx % W
    blk = (rows_*CFG["res"]//(CFG["block_km"]*1000)).astype(int)*10000 + (cols_*CFG["res"]//(CFG["block_km"]*1000)).astype(int)
    Xs = XAf[pidx]
    feats = {"bands": Xs[:, :6], "bands+ratios": Xs[:, :9], "SSL_embedding": MODELS["SSL_AE"].features(Xs),
             "bands+ratios+SSL": np.hstack([Xs[:, :9], MODELS["SSL_AE"].features(Xs)])}
    def rf(): return RandomForestClassifier(100, max_depth=12, n_jobs=-1, class_weight="balanced_subsample", random_state=0)
    out_ = []
    for fname, F in feats.items():
        for scheme in ["random", "spatial_block"]:
            oof = np.full(len(y), np.nan)
            sp = (StratifiedKFold(5, shuffle=True, random_state=0).split(F, y) if scheme == "random" else GroupKFold(5).split(F, y, blk))
            for tr, te in sp:
                if y[tr].sum() == 0: continue
                oof[te] = rf().fit(F[tr], y[tr]).predict_proba(F[te])[:, 1]
            ok_ = np.isfinite(oof)
            out_.append(dict(features=fname, cv=scheme, PR_AUC=average_precision_score(y[ok_], oof[ok_]), ROC_AUC=roc_auc_score(y[ok_], oof[ok_]),
                             Brier=brier_score_loss(y[ok_], oof[ok_]), prevalence=y[ok_].mean()))
    T3 = pd.DataFrame(out_).round(4); T3["PR_lift"] = (T3.PR_AUC/T3.prevalence).round(2); T3.to_csv(f"{OUT}/tables/T3_supervised_validation.csv", index=False)
    tr, te = next(GroupKFold(5).split(feats["bands+ratios"], y, blk))
    names = FEATS[:9]; Fm = feats["bands+ratios"]
    pi = permutation_importance(rf().fit(Fm[tr], y[tr]), Fm[te], y[te], scoring="average_precision", n_repeats=3, random_state=0, n_jobs=1)
    IMP = pd.Series(pi.importances_mean, index=names).sort_values(); IMP.to_csv(f"{OUT}/tables/T3b_permutation_importance.csv")
    print(T3.to_string())
else:
    log("No reference supplied -> supervised validation skipped (results are label-free metrics only).")

## 8. Figures

In [ ]:
def savefig(fig, name):
    p = f"{OUT}/figures/{name}.png"; fig.savefig(p, dpi=170, bbox_inches="tight"); plt.close(fig); return p
rgb = np.clip(np.nan_to_num(np.stack([data["RA"]["red"], data["RA"]["green"], data["RA"]["blue"]], -1))*3.5, 0, 1)
def show(a, ttl, cmap="viridis", q=(2, 98), ax=None):
    ax.imshow(a, cmap=cmap, vmin=np.nanpercentile(a, q[0]) if np.isfinite(a).any() else None, vmax=np.nanpercentile(a, q[1]) if np.isfinite(a).any() else None); ax.set_title(ttl); ax.axis("off")
fig, ax = plt.subplots(2, 3, figsize=(17, 11))
ax[0, 0].imshow(rgb); ax[0, 0].set_title("True colour (period A)"); ax[0, 0].axis("off")
show(np.where(validA, XA[..., 6], np.nan), "Clay ratio B11/B12", ax=ax[0, 1]); show(np.where(validA, XA[..., 7], np.nan), "Ferric ratio R/B", ax=ax[0, 2])
ax[1, 0].imshow(np.ma.masked_where(G[(P, s0, "A")] < 0, G[(P, s0, "A")]), cmap="tab10"); ax[1, 0].set_title(f"{P} clusters (A)"); ax[1, 0].axis("off")
ax[1, 1].imshow(np.ma.masked_where(G[(P, s0, "B")] < 0, G[(P, s0, "B")]), cmap="tab10"); ax[1, 1].set_title(f"{P} clusters (B, same model)"); ax[1, 1].axis("off")
ax[1, 2].imshow(mine, cmap="gray_r"); ax[1, 2].set_title("Mining-disturbance mask"); ax[1, 2].axis("off")
F1 = savefig(fig, "fig1_overview")

fig, ax = plt.subplots(1, 4, figsize=(21, 5))
for a, col, t in zip(ax, ["temporal_consistency", "seed_ARI", "spatial_coherence", "silhouette"],
                     ["Temporal consistency A→B", "Seed stability (ARI)", "Spatial coherence", "Silhouette (common space)"]):
    T1[col].plot.barh(ax=a, color=["#c44" if i == P else "#68a" for i in T1.index]); a.set_title(t)
F2 = savefig(fig, "fig2_method_comparison")

fig, ax = plt.subplots(figsize=(10, 4.5)); sns.heatmap(T2[[c for c in T2.columns if c.startswith("z_")]].rename(columns=lambda c: c[2:]), annot=True, fmt=".1f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title(f"Cluster signatures ({P}, z-scores vs. all pixels)"); F3 = savefig(fig, "fig3_cluster_signatures")

fig, ax = plt.subplots(1, 3, figsize=(18, 6))
show(AI, "Alteration index (robust z)", cmap="magma", ax=ax[0])
ax[1].imshow(conf, cmap="viridis", vmin=0, vmax=1); ax[1].set_title("Confidence (agreement across seeds × periods)"); ax[1].axis("off")
ax[2].imshow(rgb, alpha=.8); ax[2].imshow(np.ma.masked_where(~np.isin(lab_, keep), lab_ > 0), cmap="autumn", alpha=.9); ax[2].set_title(f"Exploration targets (n={len(T4)})")
for _, r in T4.head(10).iterrows():
    cy, cx = ndimage.center_of_mass(lab_ == r["id"]); ax[2].text(cx, cy, int(r["rank"]), color="cyan", fontsize=9, weight="bold")
ax[2].axis("off"); F4 = savefig(fig, "fig4_targets")

F5 = F6 = None
if T3 is not None:
    fig, ax = plt.subplots(figsize=(9, 4.5)); sns.barplot(data=T3, x="features", y="PR_AUC", hue="cv", ax=ax); ax.axhline(T3.prevalence.iloc[0], ls="--", c="k", lw=1)
    ax.set_title("PR-AUC: random vs spatial-block CV (dashed = prevalence)"); F5 = savefig(fig, "fig5_random_vs_spatial_cv")
    fig, ax = plt.subplots(figsize=(6, 4)); IMP.plot.barh(ax=ax); ax.set_title("Permutation importance (spatial hold-out)"); F6 = savefig(fig, "fig6_importance")

## 9. Auto-generated analysis + industrial report (HTML/Markdown) and ZIP

In [ ]:
def b64(p): return base64.b64encode(open(p, "rb").read()).decode() if p else ""
best = {c: T1[c].idxmax() for c in ["temporal_consistency", "seed_ARI", "spatial_coherence", "silhouette"]}
rank_P = {c: int(T1[c].rank(ascending=False)[P]) for c in best}
sent = [f"Five clustering strategies were fitted on period A ({CFG['period_A'][0]}–{CFG['period_A'][1]}, {data.get('n_scenes'+'A','?')} dates) and applied unchanged to the independent period B ({CFG['period_B'][0]}–{CFG['period_B'][1]}).",
        "Best method per label-free metric: " + "; ".join(f"{k} → {v} ({T1.loc[v, k]:.3f})" for k, v in best.items()) + ".",
        f"The proposed {P} ranks " + ", ".join(f"#{rank_P[c]} on {c}" for c in best) + f" among {len(T1)} methods. " +
        "A high rank on label-free metrics supports use only if it also wins under external validation."]
if T3 is not None:
    for f_ in T3.features.unique():
        a = T3[(T3.features == f_) & (T3.cv == "random")].PR_AUC.iloc[0]; b = T3[(T3.features == f_) & (T3.cv == "spatial_block")].PR_AUC.iloc[0]
        sent.append(f"{f_}: PR-AUC {a:.3f} (random CV) vs {b:.3f} (spatial-block CV), optimism {a-b:+.3f}.")
else:
    sent.append("No external reference was supplied, so accuracy against geology is NOT assessed; label-free metrics measure consistency, not correctness.")
analysis = " ".join(sent)
ctrl = (f"{len(T4)} candidate zones ≥ {CFG['min_target_ha']} ha were extracted where the alteration index is in the top {int((1-CFG['top_quantile'])*100)}% and "
        f"pixels were assigned to clay-rich clusters in ≥ {int(CFG['conf_min']*100)}% of runs (seeds × periods).") if len(T4) else "No zone met the target criteria; relax thresholds."
limits = ["Sentinel-2 has two SWIR bands (1610, 2190 nm): it separates clay/sericite (Al-OH) and iron-oxide surfaces but cannot identify individual minerals (alunite, pyrophyllite, chlorite, epidote) – hyperspectral data or field SWIR spectroscopy is required for that.",
          "Surface reflectance only; cover, desert varnish, dust, vegetation and mine-related disturbance can mimic or hide alteration. The disturbance mask is best-effort (OpenStreetMap/user polygons).",
          "Clusters are statistical groups; geological roles are inferred from band-ratio levels and need field confirmation.",
          "Temporal consistency below 1 reflects real change (mining activity, season) as well as model instability.",
          "Unsupervised embeddings are trained on unlabeled pixels of the whole scene (transductive); the reference labels are never used for training them."]
tbl = lambda d, n=None: (d.head(n) if n else d).to_html(classes="t", float_format=lambda x: f"{x:.3f}")
img = lambda p, c: f'<figure><img src="data:image/png;base64,{b64(p)}"><figcaption>{c}</figcaption></figure>' if p else ""
html = f"""<html><head><meta charset="utf-8"><title>Sarcheshmeh alteration study</title><style>
body{{font-family:Arial;max-width:1100px;margin:auto;padding:20px;line-height:1.5}} img{{max-width:100%}} table.t{{border-collapse:collapse;font-size:13px}}
table.t td,table.t th{{border:1px solid #bbb;padding:4px 8px}} h2{{border-bottom:2px solid #468;padding-bottom:3px}} figcaption{{font-size:12px;color:#555}}</style></head><body>
<h1>Sentinel-2 alteration mapping & target screening – Sarcheshmeh district</h1>
<p><i>Generated {time.strftime('%Y-%m-%d %H:%M')} · bbox {CFG['bbox']} · {CFG['res']} m · k={CFG['k']} · seeds={CFG['seeds']}</i></p>
<h2>1. Executive summary (industrial)</h2><p>{ctrl}</p>
<p>Use: first-pass screening to prioritise field checks. Not a resource estimate and not a substitute for geological mapping, geochemistry or drilling.</p>
<h2>2. Data and methods</h2><p>Sentinel-2 L2A (public STAC, cloud/shadow filtered with SCL, median composites per period), Copernicus DEM 30 m (slope). Features: six bands, clay ratio B11/B12, ferric ratio R/B, ferrous ratio B11/B8A, slope.
Methods: k-means on bands, on ratios, PCA(3)+k-means, and a self-supervised masked autoencoder (latent {CFG['latent']}) + k-means (with/without DEM). Mining disturbance masked ({mine.mean()*100:.2f}% of area).</p>
<h2>3. Results</h2><p>{analysis}</p>{img(F1,'Fig.1 Study overview')}<h3>Table 1 – method comparison (mean over seeds)</h3>{tbl(T1)}{img(F2,'Fig.2 Method comparison; red = proposed')}
<h3>Table 2 – cluster signatures ({P})</h3>{tbl(T2)}{img(F3,'Fig.3 Cluster z-score signatures')}
{('<h3>Table 3 – supervised validation</h3>'+tbl(T3)+img(F5,'Fig.5 Random vs spatial CV')+img(F6,'Fig.6 Importance')) if T3 is not None else '<p><b>No reference polygons supplied – supervised validation not performed.</b></p>'}
<h2>4. Exploration targets (industrial deliverable)</h2>{img(F4,'Fig.4 Alteration index, confidence and ranked targets')}{tbl(T4,15) if len(T4) else ''}
<p>Files: rasters/*.tif (cluster map, confidence, alteration index), vectors/exploration_targets.geojson, tables/*.csv. Load in QGIS/ArcGIS over geology and structure layers.</p>
<h2>5. Recommended verification protocol</h2><ol><li>Field-check top-ranked zones with a portable SWIR spectrometer (Al-OH ≈2200 nm, Mg-OH ≈2330 nm) and collect samples for XRD / thin sections.</li>
<li>Exclude zones coinciding with dumps, tailings, roads and pit walls before any decision.</li><li>Overlay faults/intrusive contacts and geochemical anomalies; rank by convergence of independent evidence.</li>
<li>Re-run with hyperspectral (EnMAP/PRISMA/EMIT) over the shortlisted zones for mineral-level mapping.</li></ol>
<h2>6. Limitations</h2><ul>{''.join(f'<li>{l}</li>' for l in limits)}</ul></body></html>"""
open(f"{OUT}/report.html", "w", encoding="utf-8").write(html)
open(f"{OUT}/report_summary.md", "w", encoding="utf-8").write("# Summary\n\n"+analysis+"\n\n"+ctrl+"\n\n## Limitations\n"+"\n".join("- "+l for l in limits))
json.dump(dict(config=CFG, python=platform.python_version(), ae_available=AE_OK, n_valid_A=int(validA.sum()), n_targets=int(len(T4)),
               metrics=T1.reset_index().to_dict("records")), open(f"{OUT}/run_metadata.json", "w"), indent=1, default=str)
open(f"{OUT}/run_log.txt", "w").write("\n".join(LOG))
shutil.make_archive("results_package", "zip", OUT)
log("saved -> results_package.zip  (open results/report.html)")
try:
    from google.colab import files; files.download("results_package.zip")
except Exception: pass